In [5]:
import pandas as pd
import numpy as np

# Load
df = pd.read_excel("uncleaneddata.xlsx")
df.columns = df.columns.str.strip()

# Convert dates
df['date']  = pd.to_datetime(df['date'],  errors='coerce')
df['pdate'] = pd.to_datetime(df['pdate'], errors='coerce')
df['ydate'] = pd.to_datetime(df['ydate'], errors='coerce')

# Temporary aligned columns
df['price_aligned'] = pd.NA
df['ytm_aligned']   = pd.NA

# Process each CUSIP separately
for cusip, g in df.groupby("cusip"):

    # Map truth dates → values
    price_map = (
        g.dropna(subset=['pdate','price'])
         .set_index('pdate')['price']
         .to_dict()
    )
    ytm_map = (
        g.dropna(subset=['ydate','ytm'])
         .set_index('ydate')['ytm']
         .to_dict()
    )

    mask = (df['cusip'] == cusip)

    # Fill aligned values based on the DATE column
    df.loc[mask, 'price_aligned'] = df.loc[mask, 'date'].map(price_map)
    df.loc[mask, 'ytm_aligned']   = df.loc[mask, 'date'].map(ytm_map)

# Overwrite original columns with aligned data
df['price'] = df['price_aligned']
df['ytm']   = df['ytm_aligned']

# Remove temp + unused columns
df = df.drop(columns=['price_aligned', 'ytm_aligned', 'pdate', 'ydate'])

# Save final aligned dataset
df.to_excel("cleaneddata.xlsx", index=False)

df.head()

,date,cusip,spread,sduration,price,ytm,coupon,maturity date
0,2024-12-06,444454AF9,1830.64,1.38,79.3,22.319,6.625,2026-08-01
1,2024-11-29,444454AF9,1579.27,1.41,82.031,19.796,6.625,2026-08-01
2,2024-11-22,444454AF9,1444.15,1.44,83.32,18.591,6.625,2026-08-01
3,2024-11-15,444454AF9,1408.32,1.46,83.597,18.236,6.625,2026-08-01
4,2024-11-08,444454AF9,1165.53,1.49,87.053,15.466,6.625,2026-08-01
